In [11]:
pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [12]:
from kaggle_secrets import UserSecretsClient
import os
from google import genai

# Kaggle ke Secrets se key uthane ka tareeqa
user_secrets = UserSecretsClient()
gemini_key = user_secrets.get_secret("GEMINI_API_KEY")

# Client initialize karein aur key pass kar dein
client = genai.Client(api_key=gemini_key)

print("Gemini client initialized via Kaggle Secrets!")

Gemini client initialized via Kaggle Secrets!


In [13]:
# Task 1.2: Make First API Call (Gemini Version)

# Simple completion call
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello and introduce yourself!',
    config={
        'max_output_tokens': 150,  # OpenAI ka max_tokens yahan max_output_tokens ban jata hai
        'temperature': 0.7
    }
)

# Extract response
message = response.text  # Gemini mein response.choices[0]... likhne ki zaroorat nahi hoti, direct .text hota hai
print(message)

Hello there!

I'm a large language model, trained by Google.


In [14]:
# Task 2.1: Create Conversation Loop (Gemini Version)

# Initialize conversation session (Gemini history ko khud handle karega)
chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        'max_output_tokens': 500,
        'temperature': 0.7
    }
)

print('Chatbot ready! Type "quit" to exit.\n')

# Conversation loop
while True:
    # Get user input
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to Gemini (history automatically manage ho rahi hai)
    response = chat_session.send_message(user_input)
         
    # Extract and print response
    assistant_response = response.text
         
    print(f'Assistant: {assistant_response}\n')

Chatbot ready! Type "quit" to exit.



You:  Hello , how are you


Assistant: Hello! I don't have feelings or a physical state like humans do, but I'm functioning perfectly and ready to assist you.

How can I help you today?



You:  how many days in a year?


Assistant: A standard year has **365 days**.

However, there's a **leap year** every four years (with some exceptions), which has **366 days**. The extra day is added to February (February 29th) to keep our calendar synchronized with the Earth's orbit around the Sun.

So, most years have 365 days, but about one in four has 366.



You:  what is the eid date in pakistan 2026?


Assistant: Predicting Eid dates for 2026, especially in Pakistan, comes with a



You:  quit


Goodbye!


In [15]:
# Task 2.2: Add System Prompt (Gemini Version)

# Define the system prompt
system_prompt = """
You are a helpful assistant.         
Be friendly, concise, and professional.         
If you don't know something, say so.
"""

# Initialize chat session with system instruction
chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": system_prompt, # OpenAI ka 'system role' yahan 'system_instruction' hai
        "temperature": 0.7,
        "max_output_tokens": 500
    }
)

print('Chatbot with System Prompt ready! Type "quit" to exit.\n')

# Baaki ka conversation loop bilkul wese hi rahega:
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to Gemini
    response = chat_session.send_message(user_input)
    assistant_response = response.text
         
    print(f'Assistant: {assistant_response}\n')

Chatbot with System Prompt ready! Type "quit" to exit.



You:  quit


Goodbye!


In [16]:
# Task 3.1: Create HR Assistant (Gemini Version)

hr_system_prompt = """
You are an HR assistant for a technology company.  
Company policies: 
- Vacation: 15 days per year 
- Sick leave: Unlimited (with manager approval) 
- Remote work: 3 days per week 
- Health insurance: Fully covered 
- 401(k) matching: Up to 5%  

Your role: 
1. Answer employee questions about policies 
2. Be friendly and supportive 
3. If unsure, suggest contacting HR department 
4. Keep responses concise (2-3 sentences)
"""

# Initialize HR chatbot session with the HR system prompt
hr_chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": hr_system_prompt,
        "temperature": 0.4,  # Kam temperature taaki policies accurate bataye
        "max_output_tokens": 300
    }
)
print('HR Assistant Ready! Ask your questions (Type "quit" to exit):\n')

# Conversation loop
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to HR Bot
    response = hr_chat_session.send_message(user_input)
    assistant_response = response.text
         
    print(f'HR Assistant: {assistant_response}\n')

HR Assistant Ready! Ask your questions (Type "quit" to exit):



You:  quit


Goodbye!


In [17]:
# Task 3.2: Create Customer Support Bot (Gemini 2.5 Flash Version)
import time
from google.genai.errors import ServerError, ClientError

support_system_prompt = """
You are a customer support agent for TechShop, an electronics retailer.  

Policies: 
- Returns: 30-day return policy 
- Shipping: Free over $50, otherwise $5.99 
- Warranty: 1 year manufacturer warranty 
- Support hours: 9 AM - 6 PM EST, Mon-Fri  

Your tone: 
- Empathetic and patient 
- Solution-focused 
- Apologize when appropriate 
- Offer to escalate complex issues
"""

# Initialize Support chatbot session using the stable 2.5-flash model
support_chat_session = client.chats.create(
    model='gemini-2.5-flash', 
    config={
        "system_instruction": support_system_prompt,
        "temperature": 0.5,
        "max_output_tokens": 300
    }
)
print('TechShop Customer Support Bot Ready! (Type "quit" to exit):\n')

# Conversation loop
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    try:
        # Send message to Support Bot
        response = support_chat_session.send_message(user_input)
        assistant_response = response.text
        print(f'Support Agent: {assistant_response}\n')
        
    except (ServerError, ClientError) as e:
        # Connection delay manage karne ke liye
        print("Support Agent: Connection is a bit busy. Please wait a moment and re-type your question!\n")
        time.sleep(2)

TechShop Customer Support Bot Ready! (Type "quit" to exit):



You:  I want to return a product I bought 2 weeks ago


Support Agent: I can certainly help you with that!

Since you purchased the product 2 weeks ago, you are well within our 30-day return policy. We want to make sure you're happy with your purchase.

To help you with the return, could you please provide your **order number**? Once I have that, I can either:

1.  **Walk you through the return process step-by-step** on our website.
2.  **Initiate the return for you** and provide instructions on how to send the item back.

Please let me know your order number, and we'll get this sorted out for you!



You:   'How much is shipping?' 


Support Agent: Thanks for asking!

Our shipping policy is:
*   **Free shipping** on all orders over $50.
*   For orders under $50, there's a flat rate of **$5.99**.

Is there anything else I can help you with regarding shipping or your return?



You:  quit


Goodbye!
